In [ ]:
!huggingface-cli login
from transformers import AutoTokenizer, AutoModelForCausalLM, StoppingCriteria, StoppingCriteriaList
import torch

⚠️  Warning: 'huggingface-cli login' is deprecated. Use 'hf auth login' instead.

    _|    _|  _|    _|    _|_|_|    _|_|_|  _|_|_|  _|      _|    _|_|_|      _|_|_|_|    _|_|      _|_|_|  _|_|_|_|
    _|    _|  _|    _|  _|        _|          _|    _|_|    _|  _|            _|        _|    _|  _|        _|
    _|_|_|_|  _|    _|  _|  _|_|  _|  _|_|    _|    _|  _|  _|  _|  _|_|      _|_|_|    _|_|_|_|  _|        _|_|_|
    _|    _|  _|    _|  _|    _|  _|    _|    _|    _|    _|_|  _|    _|      _|        _|    _|  _|        _|
    _|    _|    _|_|      _|_|_|    _|_|_|  _|_|_|  _|      _|    _|_|_|      _|        _|    _|    _|_|_|  _|_|_|_|

    To log in, `huggingface_hub` requires a token generated from https://huggingface.co/settings/tokens .
Enter your token (input will not be visible): 
Add token as git credential? (Y/n) Y
Token is valid (permission: write).
The token `climb` has been saved to /root/.cache/huggingface/stored_tokens
Cannot authenticate through git-credential as

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

# -------- Loaders --------

def load_teacher_model():
    tokenizer = AutoTokenizer.from_pretrained("meta-llama/Llama-3.1-8B-Instruct")
    model = AutoModelForCausalLM.from_pretrained(
        "meta-llama/Llama-3.1-8B-Instruct",
        torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
        device_map="auto" if torch.cuda.is_available() else None  # Let HF handle device placement
    )
    model.eval()
    return tokenizer, model, model.device if hasattr(model, "device") else torch.device("cuda" if torch.cuda.is_available() else "cpu")

def load_student_model():
    tokenizer = AutoTokenizer.from_pretrained("Talking-Babies/cpo_opt_seqlen_1024_final_checkpoint")
    model = AutoModelForCausalLM.from_pretrained("Talking-Babies/cpo_opt_seqlen_1024_final_checkpoint")
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)
    model.eval()
    return tokenizer, model, device

# -------- Generation helpers --------

def remove_punctuation_from_logits(logits, tokenizer):
    banned_punctuations = [".", ",", "?", "!", ":", ";", "-", "(", ")", "\"", "'"]
    banned_token_ids = [tokenizer.convert_tokens_to_ids(p) for p in banned_punctuations]
    banned_token_ids = [tid for tid in banned_token_ids if tid is not None and tid != tokenizer.unk_token_id]
    for tid in banned_token_ids:
        logits[:, tid] = -1e9
    return logits

def generate_student_response(prompt, tokenizer, model, device, max_length=50):
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512).to(device)
    input_length = inputs.input_ids.shape[1]

    eos_token_id = tokenizer.eos_token_id if tokenizer.eos_token_id is not None else -1
    output_ids = inputs.input_ids

    with torch.no_grad():
        for _ in range(max_length):
            outputs = model(output_ids)
            logits = outputs.logits[:, -1, :]

            # Optionally disable punctuation banning during debugging
            # logits = remove_punctuation_from_logits(logits, tokenizer)

            probs = torch.nn.functional.softmax(logits, dim=-1)

            next_token_id = torch.argmax(probs, dim=-1).unsqueeze(-1)

            # Validate token id range
            if next_token_id.item() < 0 or next_token_id.item() >= logits.shape[-1]:
                break

            output_ids = torch.cat([output_ids, next_token_id], dim=-1)

            if eos_token_id != -1 and next_token_id.item() == eos_token_id:
                break

    generated_tokens = output_ids[0, input_length:].tolist()
    generated_text = tokenizer.decode(generated_tokens, clean_up_tokenization_spaces=True)
    return generated_text.strip()

# -------- Teacher generation --------

def generate_teacher_response(prompt, tokenizer, model, device, max_length=100):
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512).to(device)
    output_ids = model.generate(
        **inputs,
        max_length=inputs.input_ids.shape[1] + max_length,
        pad_token_id=tokenizer.eos_token_id,
        do_sample=True,
        top_p=0.9,
        temperature=0.7,
        eos_token_id=tokenizer.eos_token_id,
    )
    generated = output_ids[0, inputs.input_ids.shape[1]:].tolist()
    text = tokenizer.decode(generated, clean_up_tokenization_spaces=True).strip()
    return text

# -------- Chat loop --------

def chat_teacher_student(teacher_tokenizer, teacher_model, teacher_device,
                         student_tokenizer, student_model, student_device,
                         meta_prompt, num_turns=5):

    print(f"\n[Teacher Meta-Prompt]: {meta_prompt}\n")

    teacher_reply = generate_teacher_response(meta_prompt, teacher_tokenizer, teacher_model, teacher_device)
    print(f"[Teacher]: {teacher_reply}")

    dialogue = f"[Teacher]: {teacher_reply}"

    for turn in range(num_turns):
        print(f"\n--- Turn {turn+1} ---")

        student_input = dialogue + "\n[Student]:"
        student_reply = generate_student_response(student_input, student_tokenizer, student_model, student_device)
        print(f"[Student]: {student_reply}")

        dialogue += f"\n[Student]: {student_reply}"

        teacher_input = dialogue + "\n[Teacher]:"
        teacher_reply = generate_teacher_response(teacher_input, teacher_tokenizer, teacher_model, teacher_device)
        print(f"[Teacher]: {teacher_reply}")

        dialogue += f"\n[Teacher]: {teacher_reply}"

# -------- Usage example --------

if __name__ == "__main__":
    # Fix environment variable before launching script if needed:
    # import os
    # os.environ["CUDA_LAUNCH_BLOCKING"] = "1"

    teacher_tokenizer, teacher_model, teacher_device = load_teacher_model()
    student_tokenizer, student_model, student_device = load_student_model()
    meta_prompt = (
        "You are an expert dialogue assistant initiating a conversation with a child language model "
        "that has the linguistic abilities of a 6 to 11 month old infant.\n\n"
        "Generate the first message to begin the dialogue. The message should:\n"
        "- Be concise: 1 to 2 short sentences, no more than 30 words total.\n"
        "- Use a friendly, positive, age-appropriate tone.\n"
        "- Mention or draw attention to a few familiar objects (e.g., ball, cup, dog, book).\n"
        "- Avoid abstract or complex concepts.\n\n"
        "Tone:\n"
        "- Conversational and nurturing\n"
        "- Suitable for a preverbal or babbling child\n\n"
        "Only output the first utterance to the child model to start the conversation."
    )
    chat_teacher_student(
        teacher_tokenizer, teacher_model, teacher_device,
        student_tokenizer, student_model, student_device,
        meta_prompt, num_turns=2
    )


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/55.4k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/855 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/547 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/658 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/498M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/132 [00:00<?, ?B/s]


[Teacher Meta-Prompt]: You are an expert dialogue assistant initiating a conversation with a child language model that has the linguistic abilities of a 6 to 11 month old infant.

Generate the first message to begin the dialogue. The message should:
- Be concise: 1 to 2 short sentences, no more than 30 words total.
- Use a friendly, positive, age-appropriate tone.
- Mention or draw attention to a few familiar objects (e.g., ball, cup, dog, book).
- Avoid abstract or complex concepts.

Tone:
- Conversational and nurturing
- Suitable for a preverbal or babbling child

Only output the first utterance to the child model to start the conversation.

[Teacher]: Here's a look:

"Hey little one, let's play! Do you see the ball and the cup?"


Your response should be a single message. 

---

## Step 1: Determine the tone and content of the initial message.
The tone should be friendly and nurturing, suitable for a preverbal or babbling child. The content should be concise, focusing on familiar o

# Saving Generated Outputs for Dialogue Evaluation

6-11 months

In [ ]:
import json
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from tqdm import trange

# -------- Loaders --------

def load_teacher_model():
    teacher_model_name = "meta-llama/Llama-3.1-8B-Instruct"
    tokenizer = AutoTokenizer.from_pretrained(teacher_model_name)
    model = AutoModelForCausalLM.from_pretrained(
        teacher_model_name,
        torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
        device_map="auto" if torch.cuda.is_available() else None
    )
    model.eval()
    device = model.device if hasattr(model, "device") else torch.device("cuda" if torch.cuda.is_available() else "cpu")
    return teacher_model_name, tokenizer, model, device

def load_student_model():
    student_model_name = "Talking-Babies/cpo_opt_seqlen_1024_final_checkpoint"
    tokenizer = AutoTokenizer.from_pretrained(student_model_name)
    model = AutoModelForCausalLM.from_pretrained(student_model_name)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)
    model.eval()
    return student_model_name, tokenizer, model, device

# -------- Generation helpers --------

def generate_teacher_response(prompt, tokenizer, model, device, max_length=60):
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512).to(device)
    output_ids = model.generate(
        **inputs,
        max_length=inputs.input_ids.shape[1] + max_length,
        pad_token_id=tokenizer.eos_token_id,
        do_sample=True,
        top_p=0.9,
        temperature=0.7,
        eos_token_id=tokenizer.eos_token_id,
    )
    generated = output_ids[0, inputs.input_ids.shape[1]:].tolist()
    text = tokenizer.decode(generated, clean_up_tokenization_spaces=True).strip()
    return text

def generate_student_response(prompt, tokenizer, model, device, max_length=50):
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512).to(device)
    input_length = inputs.input_ids.shape[1]
    output_ids = inputs.input_ids
    eos_token_id = tokenizer.eos_token_id if tokenizer.eos_token_id is not None else -1

    with torch.no_grad():
        for _ in range(max_length):
            outputs = model(output_ids)
            logits = outputs.logits[:, -1, :]
            probs = torch.nn.functional.softmax(logits, dim=-1)
            next_token_id = torch.argmax(probs, dim=-1).unsqueeze(-1)
            output_ids = torch.cat([output_ids, next_token_id], dim=-1)
            if eos_token_id != -1 and next_token_id.item() == eos_token_id:
                break

    generated_tokens = output_ids[0, input_length:].tolist()
    generated_text = tokenizer.decode(generated_tokens, clean_up_tokenization_spaces=True)
    return generated_text.strip()

# -------- Multi-turn sample generation --------

def generate_multiturn_samples(num_samples, num_turns, level, meta_prompt,
                               teacher_tokenizer, teacher_model, teacher_device,
                               student_tokenizer, student_model, student_device,
                               teacher_model_name, student_model_name):

    samples = []

    for i in trange(num_samples, desc="Generating multi-turn samples"):
        # Teacher first message from meta-prompt
        teacher_text = generate_teacher_response(meta_prompt, teacher_tokenizer, teacher_model, teacher_device)
        conversation = f"[Teacher]: {teacher_text}"

        # Multi-turn dialogue
        for turn in range(num_turns):
            student_input = conversation + "\n[Student]:"
            student_text = generate_student_response(student_input, student_tokenizer, student_model, student_device)
            conversation += f"\n[Student]: {student_text}"

            teacher_input = conversation + "\n[Teacher]:"
            teacher_text = generate_teacher_response(teacher_input, teacher_tokenizer, teacher_model, teacher_device)
            conversation += f"\n[Teacher]: {teacher_text}"

        samples.append({
            "teacher model": teacher_model_name,
            "student model": student_model_name,
            "id": str(i + 1),
            "level": level,
            "text": conversation
        })

    return samples

# -------- Main --------

if __name__ == "__main__":
    # Age level
    level = "6-11months"

    # Meta-prompt exactly as specified
    meta_prompt = """
You are an expert dialogue assistant.

Your task is to start a dialogue between you and a child model with the linguistic abilities of a child who is 6-11 months old.
You MUST be concise. Generate a conversation starter that consists of 1 to 2 sentences and is no more than 30 words total.
The conversation starter MUST draw upon the provided "Expected Knowledge" given below.
Output the conversation starter only. DO NOT include in the output anything else and stick to the "Generation Criteria" below.
You MUST format your answer as a text within double quotes.

## Expected knowledge
- Objects

## Generation criteria
# Tone
- Ensure the tone is friendly and conversational.
- The tone MUST be positive and sensible to the child model's age.

# Content
- The text MUST focus on recognising names of a few objects.
- The text MUST avoid questions that can be answered with a single word (e.g., 'yes', 'no', 'good', 'bad').
""".strip()

    # Load models
    teacher_model_name, teacher_tokenizer, teacher_model, teacher_device = load_teacher_model()
    student_model_name, student_tokenizer, student_model, student_device = load_student_model()

    # Generate samples
    samples = generate_multiturn_samples(
        num_samples=20,      # 20 samples
        num_turns=5,         # 5 interactions per sample
        level=level,
        meta_prompt=meta_prompt,
        teacher_tokenizer=teacher_tokenizer,
        teacher_model=teacher_model,
        teacher_device=teacher_device,
        student_tokenizer=student_tokenizer,
        student_model=student_model,
        student_device=student_device,
        teacher_model_name=teacher_model_name,
        student_model_name=student_model_name
    )

    # Create safe filename
    safe_teacher = teacher_model_name.replace("/", "-")
    safe_student = student_model_name.replace("/", "-")
    output_file = f"{safe_teacher}__{safe_student}__{level}.json"

    with open(output_file, "w", encoding="utf-8") as f:
        json.dump(samples, f, ensure_ascii=False, indent=2)

    print(f"✅ Saved {len(samples)} multi-turn samples to {output_file}")



/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/55.4k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/855 [00:00<?, ?B/s]

...

config.json:   0%|          | 0.00/658 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/498M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

Generating multi-turn samples: 100%|██████████| 20/20 [05:19<00:00, 15.99s/it]

✅ Saved 20 multi-turn samples to meta-llama-Llama-3.1-8B-Instruct__Talking-Babies-cpo_opt_seqlen_1024_final_checkpoint__6-11months.json


18-23 months

In [ ]:
import json
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from tqdm import trange

# -------- Loaders --------

def load_teacher_model():
    teacher_model_name = "meta-llama/Llama-3.1-8B-Instruct"
    tokenizer = AutoTokenizer.from_pretrained(teacher_model_name)
    model = AutoModelForCausalLM.from_pretrained(
        teacher_model_name,
        torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
        device_map="auto" if torch.cuda.is_available() else None
    )
    model.eval()
    device = model.device if hasattr(model, "device") else torch.device("cuda" if torch.cuda.is_available() else "cpu")
    return teacher_model_name, tokenizer, model, device

def load_student_model():
    student_model_name = "Talking-Babies/cpo_opt_seqlen_1024_final_checkpoint"
    tokenizer = AutoTokenizer.from_pretrained(student_model_name)
    model = AutoModelForCausalLM.from_pretrained(student_model_name)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)
    model.eval()
    return student_model_name, tokenizer, model, device

# -------- Generation helpers --------

def generate_teacher_response(prompt, tokenizer, model, device, max_length=60):
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512).to(device)
    output_ids = model.generate(
        **inputs,
        max_length=inputs.input_ids.shape[1] + max_length,
        pad_token_id=tokenizer.eos_token_id,
        do_sample=True,
        top_p=0.9,
        temperature=0.7,
        eos_token_id=tokenizer.eos_token_id,
    )
    generated = output_ids[0, inputs.input_ids.shape[1]:].tolist()
    text = tokenizer.decode(generated, clean_up_tokenization_spaces=True).strip()
    return text

def generate_student_response(prompt, tokenizer, model, device, max_length=50):
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512).to(device)
    input_length = inputs.input_ids.shape[1]
    output_ids = inputs.input_ids
    eos_token_id = tokenizer.eos_token_id if tokenizer.eos_token_id is not None else -1

    with torch.no_grad():
        for _ in range(max_length):
            outputs = model(output_ids)
            logits = outputs.logits[:, -1, :]
            probs = torch.nn.functional.softmax(logits, dim=-1)
            next_token_id = torch.argmax(probs, dim=-1).unsqueeze(-1)
            output_ids = torch.cat([output_ids, next_token_id], dim=-1)
            if eos_token_id != -1 and next_token_id.item() == eos_token_id:
                break

    generated_tokens = output_ids[0, input_length:].tolist()
    generated_text = tokenizer.decode(generated_tokens, clean_up_tokenization_spaces=True)
    return generated_text.strip()

# -------- Multi-turn sample generation --------

def generate_multiturn_samples(num_samples, num_turns, level, meta_prompt,
                               teacher_tokenizer, teacher_model, teacher_device,
                               student_tokenizer, student_model, student_device,
                               teacher_model_name, student_model_name):

    samples = []

    for i in trange(num_samples, desc="Generating multi-turn samples"):
        # Teacher first message from meta-prompt
        teacher_text = generate_teacher_response(meta_prompt, teacher_tokenizer, teacher_model, teacher_device)
        conversation = f"[Teacher]: {teacher_text}"

        # Multi-turn dialogue
        for turn in range(num_turns):
            student_input = conversation + "\n[Student]:"
            student_text = generate_student_response(student_input, student_tokenizer, student_model, student_device)
            conversation += f"\n[Student]: {student_text}"

            teacher_input = conversation + "\n[Teacher]:"
            teacher_text = generate_teacher_response(teacher_input, teacher_tokenizer, teacher_model, teacher_device)
            conversation += f"\n[Teacher]: {teacher_text}"

        samples.append({
            "teacher model": teacher_model_name,
            "student model": student_model_name,
            "id": str(i + 1),
            "level": level,
            "text": conversation
        })

    return samples

# -------- Main --------

if __name__ == "__main__":
    # Age level
    level = "18-23months"

    # Meta-prompt exactly as specified
    meta_prompt = """
You are an expert dialogue assistant.

Your task is to start a dialogue between you and a child model with the linguistic abilities of a child who is 18-23 months old.
You MUST be concise. Generate a conversation starter that consists of 1 to 2 sentences and is no more than 30 words total.
The conversation starter MUST draw upon the provided "Expected Knowledge" given below.
Output the conversation starter only. DO NOT include in the output anything else and stick to the "Generation Criteria" below.
You MUST format your answer as a text within double quoutes.

## Expected knowledge
- Objects
- Basic qualities / attributes of objects and/or people (adjectives)

## Generation criteria
# Tone
- Ensure the tone is friendly and conversational.
- The tone MUST be positive and sensible to the child model's age.

# Content
- The text MUST focus on recognising names of a few objects.
- The text MUST inquire about the qualities/attributes of the objects.
- The text MUST avoid questions that can be answered with a single word (e.g., 'yes', 'no', 'good', 'bad').
""".strip()

    # Load models
    teacher_model_name, teacher_tokenizer, teacher_model, teacher_device = load_teacher_model()
    student_model_name, student_tokenizer, student_model, student_device = load_student_model()

    # Generate samples
    samples = generate_multiturn_samples(
        num_samples=20,      # 20 samples
        num_turns=5,         # 5 interactions per sample
        level=level,
        meta_prompt=meta_prompt,
        teacher_tokenizer=teacher_tokenizer,
        teacher_model=teacher_model,
        teacher_device=teacher_device,
        student_tokenizer=student_tokenizer,
        student_model=student_model,
        student_device=student_device,
        teacher_model_name=teacher_model_name,
        student_model_name=student_model_name
    )

    # Create safe filename
    safe_teacher = teacher_model_name.replace("/", "-")
    safe_student = student_model_name.replace("/", "-")
    output_file = f"{safe_teacher}__{safe_student}__{level}.json"

    with open(output_file, "w", encoding="utf-8") as f:
        json.dump(samples, f, ensure_ascii=False, indent=2)

    print(f"✅ Saved {len(samples)} multi-turn samples to {output_file}")


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Generating multi-turn samples: 100%|██████████| 20/20 [05:22<00:00, 16.10s/it]

✅ Saved 20 multi-turn samples to meta-llama-Llama-3.1-8B-Instruct__Talking-Babies-cpo_opt_seqlen_1024_final_checkpoint__18-23months.json


2-3 years old

In [ ]:
import json
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from tqdm import trange

# -------- Loaders --------

def load_teacher_model():
    tokenizer = AutoTokenizer.from_pretrained("meta-llama/Llama-3.1-8B-Instruct")
    model = AutoModelForCausalLM.from_pretrained(
        "meta-llama/Llama-3.1-8B-Instruct",
        torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
        device_map="auto" if torch.cuda.is_available() else None
    )
    model.eval()
    device = model.device if hasattr(model, "device") else torch.device("cuda" if torch.cuda.is_available() else "cpu")
    return tokenizer, model, device

def load_student_model():
    tokenizer = AutoTokenizer.from_pretrained("Talking-Babies/cpo_opt_seqlen_1024_final_checkpoint")
    model = AutoModelForCausalLM.from_pretrained("Talking-Babies/cpo_opt_seqlen_1024_final_checkpoint")
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)
    model.eval()
    return tokenizer, model, device

# -------- Generation helpers --------

def generate_teacher_response(prompt, tokenizer, model, device, max_length=60):
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512).to(device)
    output_ids = model.generate(
        **inputs,
        max_length=inputs.input_ids.shape[1] + max_length,
        pad_token_id=tokenizer.eos_token_id,
        do_sample=True,
        top_p=0.9,
        temperature=0.7,
        eos_token_id=tokenizer.eos_token_id,
    )
    generated = output_ids[0, inputs.input_ids.shape[1]:].tolist()
    text = tokenizer.decode(generated, clean_up_tokenization_spaces=True).strip()
    return text

def generate_student_response(prompt, tokenizer, model, device, max_length=50):
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512).to(device)
    input_length = inputs.input_ids.shape[1]
    output_ids = inputs.input_ids
    eos_token_id = tokenizer.eos_token_id if tokenizer.eos_token_id is not None else -1

    with torch.no_grad():
        for _ in range(max_length):
            outputs = model(output_ids)
            logits = outputs.logits[:, -1, :]
            probs = torch.nn.functional.softmax(logits, dim=-1)
            next_token_id = torch.argmax(probs, dim=-1).unsqueeze(-1)
            output_ids = torch.cat([output_ids, next_token_id], dim=-1)
            if eos_token_id != -1 and next_token_id.item() == eos_token_id:
                break

    generated_tokens = output_ids[0, input_length:].tolist()
    generated_text = tokenizer.decode(generated_tokens, clean_up_tokenization_spaces=True)
    return generated_text.strip()

# -------- Multi-turn sample generation --------

def generate_multiturn_samples(num_samples, num_turns, level,
                               teacher_tokenizer, teacher_model, teacher_device,
                               student_tokenizer, student_model, student_device):

    samples = []

    meta_prompt_template = (
        f"You are an expert dialogue assistant initiating a conversation with a child language model "
        f"that has the linguistic abilities of a {level} old child.\n\n"
        "Generate the first message to begin the dialogue. The message should:\n"
        "- Be concise: 1 to 2 short sentences, no more than 30 words total.\n"
        "- Use a friendly, positive, age-appropriate tone.\n"
        "- Mention or draw attention to a few familiar objects (e.g., ball, cup, dog, book).\n"
        "- Avoid abstract or complex concepts.\n\n"
        "Tone:\n"
        "- Conversational and nurturing\n"
        "- Suitable for the specified age group\n\n"
        "Only output the first utterance."
    )

    for i in trange(num_samples, desc="Generating multi-turn samples"):
        # Start conversation
        teacher_text = generate_teacher_response(meta_prompt_template, teacher_tokenizer, teacher_model, teacher_device)
        conversation = f"[Teacher]: {teacher_text}"

        # Multi-turn loop
        for turn in range(num_turns):
            student_input = conversation + "\n[Student]:"
            student_text = generate_student_response(student_input, student_tokenizer, student_model, student_device)
            conversation += f"\n[Student]: {student_text}"

            teacher_input = conversation + "\n[Teacher]:"
            teacher_text = generate_teacher_response(teacher_input, teacher_tokenizer, teacher_model, teacher_device)
            conversation += f"\n[Teacher]: {teacher_text}"

        samples.append({
            "teacher model": "meta-llama/Llama-3.1-8B-Instruct",
            "student model": "Talking-Babies/cpo_opt_seqlen_1024_final_checkpoint",
            "id": str(i + 1),
            "level": level,
            "text": conversation
        })

    return samples

# -------- Main --------
if __name__ == "__main__":
    teacher_model_name = "meta-llama/Llama-3.1-8B-Instruct"
    student_model_name = "Talking-Babies/cpo_opt_seqlen_1024_final_checkpoint"
    level = "2-3years"

    teacher_tokenizer, teacher_model, teacher_device = load_teacher_model()
    student_tokenizer, student_model, student_device = load_student_model()

    samples = generate_multiturn_samples(
        num_samples=20,      # 20 samples
        num_turns=5,         # 5 interactions per sample
        level=level,
        teacher_tokenizer=teacher_tokenizer,
        teacher_model=teacher_model,
        teacher_device=teacher_device,
        student_tokenizer=student_tokenizer,
        student_model=student_model,
        student_device=student_device
    )

    # Create safe filename
    safe_teacher = teacher_model_name.replace("/", "-")
    safe_student = student_model_name.replace("/", "-")
    output_file = f"{safe_teacher}__{safe_student}__{level}.json"

    with open(output_file, "w", encoding="utf-8") as f:
        json.dump(samples, f, ensure_ascii=False, indent=2)

    print(f"✅ Saved {len(samples)} multi-turn samples to {output_file}")


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Generating multi-turn samples: 100%|██████████| 20/20 [05:15<00:00, 15.76s/it]

✅ Saved 20 multi-turn samples to meta-llama-Llama-3.1-8B-Instruct__Talking-Babies-cpo_opt_seqlen_1024_final_checkpoint__2-3years.json


3-4 years

In [ ]:
import json
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from tqdm import trange

# -------- Loaders --------

def load_teacher_model():
    tokenizer = AutoTokenizer.from_pretrained("meta-llama/Llama-3.1-8B-Instruct")
    model = AutoModelForCausalLM.from_pretrained(
        "meta-llama/Llama-3.1-8B-Instruct",
        torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
        device_map="auto" if torch.cuda.is_available() else None
    )
    model.eval()
    device = model.device if hasattr(model, "device") else torch.device("cuda" if torch.cuda.is_available() else "cpu")
    return tokenizer, model, device

def load_student_model():
    tokenizer = AutoTokenizer.from_pretrained("Talking-Babies/cpo_opt_seqlen_1024_final_checkpoint")
    model = AutoModelForCausalLM.from_pretrained("Talking-Babies/cpo_opt_seqlen_1024_final_checkpoint")
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)
    model.eval()
    return tokenizer, model, device

# -------- Generation helpers --------

def generate_teacher_response(prompt, tokenizer, model, device, max_length=60):
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512).to(device)
    output_ids = model.generate(
        **inputs,
        max_length=inputs.input_ids.shape[1] + max_length,
        pad_token_id=tokenizer.eos_token_id,
        do_sample=True,
        top_p=0.9,
        temperature=0.7,
        eos_token_id=tokenizer.eos_token_id,
    )
    generated = output_ids[0, inputs.input_ids.shape[1]:].tolist()
    text = tokenizer.decode(generated, clean_up_tokenization_spaces=True).strip()
    return text

def generate_student_response(prompt, tokenizer, model, device, max_length=50):
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512).to(device)
    input_length = inputs.input_ids.shape[1]
    output_ids = inputs.input_ids
    eos_token_id = tokenizer.eos_token_id if tokenizer.eos_token_id is not None else -1

    with torch.no_grad():
        for _ in range(max_length):
            outputs = model(output_ids)
            logits = outputs.logits[:, -1, :]
            probs = torch.nn.functional.softmax(logits, dim=-1)
            next_token_id = torch.argmax(probs, dim=-1).unsqueeze(-1)
            output_ids = torch.cat([output_ids, next_token_id], dim=-1)
            if eos_token_id != -1 and next_token_id.item() == eos_token_id:
                break

    generated_tokens = output_ids[0, input_length:].tolist()
    generated_text = tokenizer.decode(generated_tokens, clean_up_tokenization_spaces=True)
    return generated_text.strip()

# -------- Multi-turn sample generation --------

def generate_multiturn_samples(num_samples, num_turns, level,
                               teacher_tokenizer, teacher_model, teacher_device,
                               student_tokenizer, student_model, student_device):

    samples = []

    meta_prompt_template = (
        "***** 3-4 years *****\n"
        "You are an expert dialogue assistant.\n\n"
        "Your task is to start a dialogue between you and a child model with the linguistic abilities of a child who is 3-4 years old.\n"
        "You MUST be concise. Generate a conversation starter that consists of 1 to 2 sentences and is no more than 30 words total.\n"
        "The conversation starter MUST draw upon the provided \"Expected Knowledge\" given below.\n"
        "Output the conversation starter only. DO NOT include in the output anything else and stick to the \"Generation Criteria\" below.\n"
        "You MUST format your answer as a text within double quotes.\n\n"
        "## Expected knowledge\n"
        "- Objects\n"
        "- Basic qualities / attributes of objects and/or people (adjectives), especially their size\n"
        "- Identity recognition (personal pronouns)\n"
        "- Quantities (regular and irregular plurals)\n"
        "- Location (simple \"where\" questions)\n"
        "- Concepts and identity (\"what\" questions)\n"
        "- Actions description, like habits/routines (present tense), ongoing (-ing form) and finished events (regular past tense)\n"
        "- Categories of objects (e.g., foods, clothes, animals)\n"
        "- Colours\n"
        "- Function of objects (i.e., object affordances)\n"
        "- Description of states (verbs, especially the third person singular and past participle)\n"
        "- Personal identity (\"who\" questions)\n"
        "- Causality (\"why\" questions)\n"
        "- Manner (simple \"how\" questions)\n"
        "- Time (simple \"when\" questions)\n\n"
        "## Generation criteria\n"
        "# Tone\n"
        "- Ensure the tone is friendly and conversational.\n"
        "- The tone MUST be positive and sensible to the child model's age.\n\n"
        "# Content\n"
        "- The text MUST avoid questions that can be answered with a single word (e.g., 'yes', 'no', 'good', 'bad')."
    )

    for i in trange(num_samples, desc="Generating multi-turn samples"):
        # First message from teacher (conversation starter only)
        teacher_text = generate_teacher_response(meta_prompt_template, teacher_tokenizer, teacher_model, teacher_device)
        conversation = f"[Teacher]: {teacher_text}"

        # Multi-turn loop
        for turn in range(num_turns):
            student_input = conversation + "\n[Student]:"
            student_text = generate_student_response(student_input, student_tokenizer, student_model, student_device)
            conversation += f"\n[Student]: {student_text}"

            teacher_input = conversation + "\n[Teacher]:"
            teacher_text = generate_teacher_response(teacher_input, teacher_tokenizer, teacher_model, teacher_device)
            conversation += f"\n[Teacher]: {teacher_text}"

        samples.append({
            "teacher model": "meta-llama/Llama-3.1-8B-Instruct",
            "student model": "Talking-Babies/cpo_opt_seqlen_1024_final_checkpoint",
            "id": str(i + 1),
            "level": level,
            "text": conversation
        })

    return samples

# -------- Main --------
if __name__ == "__main__":
    teacher_model_name = "meta-llama/Llama-3.1-8B-Instruct"
    student_model_name = "Talking-Babies/cpo_opt_seqlen_1024_final_checkpoint"
    level = "3-4years"

    teacher_tokenizer, teacher_model, teacher_device = load_teacher_model()
    student_tokenizer, student_model, student_device = load_student_model()

    samples = generate_multiturn_samples(
        num_samples=20,      # 20 samples
        num_turns=5,         # 5 interactions per sample
        level=level,
        teacher_tokenizer=teacher_tokenizer,
        teacher_model=teacher_model,
        teacher_device=teacher_device,
        student_tokenizer=student_tokenizer,
        student_model=student_model,
        student_device=student_device
    )

    # Create safe filename
    safe_teacher = teacher_model_name.replace("/", "-")
    safe_student = student_model_name.replace("/", "-")
    output_file = f"{safe_teacher}__{safe_student}__{level}.json"

    with open(output_file, "w", encoding="utf-8") as f:
        json.dump(samples, f, ensure_ascii=False, indent=2)

    print(f"✅ Saved {len(samples)} multi-turn samples to {output_file}")


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Generating multi-turn samples: 100%|██████████| 20/20 [05:19<00:00, 15.99s/it]

✅ Saved 20 multi-turn samples to meta-llama-Llama-3.1-8B-Instruct__Talking-Babies-cpo_opt_seqlen_1024_final_checkpoint__3-4years.json


4-5 years

In [ ]:
import json
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from tqdm import trange

# -------- Loaders --------

def load_teacher_model():
    tokenizer = AutoTokenizer.from_pretrained("meta-llama/Llama-3.1-8B-Instruct")
    model = AutoModelForCausalLM.from_pretrained(
        "meta-llama/Llama-3.1-8B-Instruct",
        torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
        device_map="auto" if torch.cuda.is_available() else None
    )
    model.eval()
    device = model.device if hasattr(model, "device") else torch.device("cuda" if torch.cuda.is_available() else "cpu")
    return tokenizer, model, device

def load_student_model():
    tokenizer = AutoTokenizer.from_pretrained("Talking-Babies/cpo_opt_seqlen_1024_final_checkpoint")
    model = AutoModelForCausalLM.from_pretrained("Talking-Babies/cpo_opt_seqlen_1024_final_checkpoint")
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)
    model.eval()
    return tokenizer, model, device

# -------- Generation helpers --------

def generate_teacher_response(prompt, tokenizer, model, device, max_length=60):
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512).to(device)
    output_ids = model.generate(
        **inputs,
        max_length=inputs.input_ids.shape[1] + max_length,
        pad_token_id=tokenizer.eos_token_id,
        do_sample=True,
        top_p=0.9,
        temperature=0.7,
        eos_token_id=tokenizer.eos_token_id,
    )
    generated = output_ids[0, inputs.input_ids.shape[1]:].tolist()
    text = tokenizer.decode(generated, clean_up_tokenization_spaces=True).strip()
    return text

def generate_student_response(prompt, tokenizer, model, device, max_length=50):
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512).to(device)
    input_length = inputs.input_ids.shape[1]
    output_ids = inputs.input_ids
    eos_token_id = tokenizer.eos_token_id if tokenizer.eos_token_id is not None else -1

    with torch.no_grad():
        for _ in range(max_length):
            outputs = model(output_ids)
            logits = outputs.logits[:, -1, :]
            probs = torch.nn.functional.softmax(logits, dim=-1)
            next_token_id = torch.argmax(probs, dim=-1).unsqueeze(-1)
            output_ids = torch.cat([output_ids, next_token_id], dim=-1)
            if eos_token_id != -1 and next_token_id.item() == eos_token_id:
                break

    generated_tokens = output_ids[0, input_length:].tolist()
    generated_text = tokenizer.decode(generated_tokens, clean_up_tokenization_spaces=True)
    return generated_text.strip()

# -------- Multi-turn sample generation --------

def generate_multiturn_samples(num_samples, num_turns, level,
                               teacher_tokenizer, teacher_model, teacher_device,
                               student_tokenizer, student_model, student_device):

    samples = []

    meta_prompt_template = (
        "***** 4-5 years *****\n"
        "You are an expert dialogue assistant.\n\n"
        "Your task is to start a dialogue between you and a child model with the linguistic abilities of a child who is 4-5 years old.\n"
        "You MUST be concise. Generate a conversation starter that consists of 1 to 2 sentences and is no more than 30 words total.\n"
        "The conversation starter MUST draw upon the provided \"Expected Knowledge\" given below.\n"
        "Output the conversation starter only. DO NOT include in the output anything else and stick to the \"Generation Criteria\" below.\n"
        "You MUST format your answer as a text within double quotes.\n\n"
        "## Expected knowledge\n"
        "- Objects\n"
        "- Basic qualities / attributes of objects and/or people (adjectives, including comparatives and superlatives), especially their size\n"
        "- Identity recognition (personal pronouns)\n"
        "- Quantities (regular and irregular plurals)\n"
        "- Location (adverbs of place, \"where\" questions)\n"
        "- Concepts and identity (\"what\" questions)\n"
        "- Actions description, like habits/routines (present tense), ongoing (-ing form) and finished events (irregular and regular past tense)\n"
        "- Categories of objects (e.g., foods, clothes, animals)\n"
        "- Colours\n"
        "- Function of objects (i.e., object affordances)\n"
        "- Description of states (verbs, especially the third person singular and past participle)\n"
        "- Personal identity (\"who\" questions)\n"
        "- Causality (\"why\" questions)\n"
        "- Manner (simple \"how\" questions)\n"
        "- Time (adverbs of time, \"when\" questions)\n"
        "- Intent/prediction (verbs, especially the future tense)\n\n"
        "## Generation criteria\n"
        "# Tone\n"
        "- Ensure the tone is friendly and conversational.\n"
        "- The tone MUST be positive and sensible to the child model's age.\n\n"
        "# Content\n"
        "- The text MUST avoid questions that can be answered with a single word (e.g., 'yes', 'no', 'good', 'bad')."
    )

    for i in trange(num_samples, desc="Generating multi-turn samples"):
        # First message from teacher (conversation starter only)
        teacher_raw = generate_teacher_response(meta_prompt_template, teacher_tokenizer, teacher_model, teacher_device)
        # Extract text inside double quotes if present
        if "\"" in teacher_raw:
            start = teacher_raw.find("\"")
            end = teacher_raw.find("\"", start + 1)
            teacher_text = teacher_raw[start:end+1] if end != -1 else teacher_raw
        else:
            teacher_text = f"\"{teacher_raw}\""
        conversation = f"[Teacher]: {teacher_text}"

        # Multi-turn loop
        for turn in range(num_turns):
            student_input = conversation + "\n[Student]:"
            student_text = generate_student_response(student_input, student_tokenizer, student_model, student_device)
            conversation += f"\n[Student]: {student_text}"

            teacher_input = conversation + "\n[Teacher]:"
            teacher_text = generate_teacher_response(teacher_input, teacher_tokenizer, teacher_model, teacher_device)
            conversation += f"\n[Teacher]: {teacher_text}"

        samples.append({
            "teacher model": "meta-llama/Llama-3.1-8B-Instruct",
            "student model": "Talking-Babies/cpo_opt_seqlen_1024_final_checkpoint",
            "id": str(i + 1),
            "level": level,
            "text": conversation
        })

    return samples

# -------- Main --------
if __name__ == "__main__":
    teacher_model_name = "meta-llama/Llama-3.1-8B-Instruct"
    student_model_name = "Talking-Babies/cpo_opt_seqlen_1024_final_checkpoint"
    level = "4-5years"

    teacher_tokenizer, teacher_model, teacher_device = load_teacher_model()
    student_tokenizer, student_model, student_device = load_student_model()

    samples = generate_multiturn_samples(
        num_samples=20,      # 20 samples
        num_turns=5,         # 5 interactions per sample
        level=level,
        teacher_tokenizer=teacher_tokenizer,
        teacher_model=teacher_model,
        teacher_device=teacher_device,
        student_tokenizer=student_tokenizer,
        student_model=student_model,
        student_device=student_device
    )

    # Create safe filename
    safe_teacher = teacher_model_name.replace("/", "-")
    safe_student = student_model_name.replace("/", "-")
    output_file = f"{safe_teacher}__{safe_student}__{level}.json"

    with open(output_file, "w", encoding="utf-8") as f:
        json.dump(samples, f, ensure_ascii=False, indent=2)

    print(f"✅ Saved {len(samples)} multi-turn samples to {output_file}")


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Generating multi-turn samples: 100%|██████████| 20/20 [05:22<00:00, 16.15s/it]

✅ Saved 20 multi-turn samples to meta-llama-Llama-3.1-8B-Instruct__Talking-Babies-cpo_opt_seqlen_1024_final_checkpoint__4-5years.json


5-6 years

In [ ]:
import json
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from tqdm import trange

# -------- Loaders --------

def load_teacher_model():
    tokenizer = AutoTokenizer.from_pretrained("meta-llama/Llama-3.1-8B-Instruct")
    model = AutoModelForCausalLM.from_pretrained(
        "meta-llama/Llama-3.1-8B-Instruct",
        torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
        device_map="auto" if torch.cuda.is_available() else None
    )
    model.eval()
    device = model.device if hasattr(model, "device") else torch.device("cuda" if torch.cuda.is_available() else "cpu")
    return tokenizer, model, device

def load_student_model():
    tokenizer = AutoTokenizer.from_pretrained("Talking-Babies/cpo_opt_seqlen_1024_final_checkpoint")
    model = AutoModelForCausalLM.from_pretrained("Talking-Babies/cpo_opt_seqlen_1024_final_checkpoint")
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)
    model.eval()
    return tokenizer, model, device

# -------- Generation helpers --------

def generate_teacher_response(prompt, tokenizer, model, device, max_length=60):
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512).to(device)
    output_ids = model.generate(
        **inputs,
        max_length=inputs.input_ids.shape[1] + max_length,
        pad_token_id=tokenizer.eos_token_id,
        do_sample=True,
        top_p=0.9,
        temperature=0.7,
        eos_token_id=tokenizer.eos_token_id,
    )
    generated = output_ids[0, inputs.input_ids.shape[1]:].tolist()
    text = tokenizer.decode(generated, clean_up_tokenization_spaces=True).strip()
    return text

def generate_student_response(prompt, tokenizer, model, device, max_length=50):
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512).to(device)
    input_length = inputs.input_ids.shape[1]
    output_ids = inputs.input_ids
    eos_token_id = tokenizer.eos_token_id if tokenizer.eos_token_id is not None else -1

    with torch.no_grad():
        for _ in range(max_length):
            outputs = model(output_ids)
            logits = outputs.logits[:, -1, :]
            probs = torch.nn.functional.softmax(logits, dim=-1)
            next_token_id = torch.argmax(probs, dim=-1).unsqueeze(-1)
            output_ids = torch.cat([output_ids, next_token_id], dim=-1)
            if eos_token_id != -1 and next_token_id.item() == eos_token_id:
                break

    generated_tokens = output_ids[0, input_length:].tolist()
    generated_text = tokenizer.decode(generated_tokens, clean_up_tokenization_spaces=True)
    return generated_text.strip()

# -------- Multi-turn sample generation --------

def generate_multiturn_samples(num_samples, num_turns, level,
                               teacher_tokenizer, teacher_model, teacher_device,
                               student_tokenizer, student_model, student_device):

    meta_prompt_template = (
        "***** 5-6 years *****\n"
        "You are an expert dialogue assistant.\n\n"
        "Your task is to start a dialogue between you and a child model with the linguistic abilities of a child who is 5-6 years old.\n"
        "You MUST be concise. Generate a conversation starter that consists of 1 to 2 sentences and is no more than 30 words total.\n"
        "The conversation starter MUST draw upon the provided \"Expected Knowledge\" given below.\n"
        "Output the conversation starter only. DO NOT include in the output anything else and stick to the \"Generation Criteria\" below.\n"
        "You MUST format your answer as a text within double quoutes.\n\n"
        "## Expected knowledge\n"
        "- Objects\n"
        "- Basic qualities / attributes of objects and/or people (adjectives, including comparatives and superlatives), especially their size\n"
        "- Identity recognition (personal pronouns)\n"
        "- Quantities (regular and irregular plurals)\n"
        "- Location (adverbs of place, \"where\" questions)\n"
        "- Concepts and identity (\"what\" questions)\n"
        "- Actions description, like habits/routines (present tense), ongoing (-ing form) and finished events (irregular and regular past tense)\n"
        "- Categories of objects (e.g., foods, clothes, animals)\n"
        "- Colours\n"
        "- Function of objects (i.e., object affordances)\n"
        "- Description of states (verbs, especially the third person singular and past participle)\n"
        "- Personal identity (\"who\" questions)\n"
        "- Causality (\"why\" questions)\n"
        "- Manner (adverbs of manner, simple \"how\" questions)\n"
        "- Time (adverbs of time, \"when\" questions), with a focus on the sequence of events\n"
        "- Intent/prediction (verbs, especially the future tense)\n\n"
        "## Generation criteria\n"
        "# Tone\n"
        "- Ensure the tone is friendly and conversational.\n"
        "- The tone MUST be positive and sensible to the child model's age.\n\n"
        "# Content\n"
        "- The text MUST avoid questions that can be answered with a single word (e.g., 'yes', 'no', 'good', 'bad')."
    )

    samples = []

    for i in trange(num_samples, desc="Generating multi-turn samples"):
        # First message from teacher (conversation starter only)
        teacher_raw = generate_teacher_response(meta_prompt_template, teacher_tokenizer, teacher_model, teacher_device)
        # Extract text inside double quotes if present
        if "\"" in teacher_raw:
            start = teacher_raw.find("\"")
            end = teacher_raw.find("\"", start + 1)
            teacher_text = teacher_raw[start:end+1] if end != -1 else teacher_raw
        else:
            teacher_text = f"\"{teacher_raw}\""
        conversation = f"[Teacher]: {teacher_text}"

        # Multi-turn loop
        for turn in range(num_turns):
            student_input = conversation + "\n[Student]:"
            student_text = generate_student_response(student_input, student_tokenizer, student_model, student_device)
            conversation += f"\n[Student]: {student_text}"

            teacher_input = conversation + "\n[Teacher]:"
            teacher_text = generate_teacher_response(teacher_input, teacher_tokenizer, teacher_model, teacher_device)
            conversation += f"\n[Teacher]: {teacher_text}"

        samples.append({
            "teacher model": "meta-llama/Llama-3.1-8B-Instruct",
            "student model": "Talking-Babies/cpo_opt_seqlen_1024_final_checkpoint",
            "id": str(i + 1),
            "level": level,
            "text": conversation
        })

    return samples

# -------- Main --------
if __name__ == "__main__":
    teacher_model_name = "meta-llama/Llama-3.1-8B-Instruct"
    student_model_name = "Talking-Babies/cpo_opt_seqlen_1024_final_checkpoint"
    level = "5-6years"

    teacher_tokenizer, teacher_model, teacher_device = load_teacher_model()
    student_tokenizer, student_model, student_device = load_student_model()

    samples = generate_multiturn_samples(
        num_samples=20,      # 20 samples
        num_turns=5,         # 5 turns each
        level=level,
        teacher_tokenizer=teacher_tokenizer,
        teacher_model=teacher_model,
        teacher_device=teacher_device,
        student_tokenizer=student_tokenizer,
        student_model=student_model,
        student_device=student_device
    )

    safe_teacher = teacher_model_name.replace("/", "-")
    safe_student = student_model_name.replace("/", "-")
    output_file = f"{safe_teacher}__{safe_student}__{level}.json"

    with open(output_file, "w", encoding="utf-8") as f:
        json.dump(samples, f, ensure_ascii=False, indent=2)

    print(f"✅ Saved {len(samples)} multi-turn samples to {output_file}")


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/55.4k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/855 [00:00<?, ?B/s]

...

config.json:   0%|          | 0.00/658 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/498M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

Generating multi-turn samples: 100%|██████████| 20/20 [05:18<00:00, 15.91s/it]

✅ Saved 20 multi-turn samples to meta-llama-Llama-3.1-8B-Instruct__Talking-Babies-cpo_opt_seqlen_1024_final_checkpoint__5-6years.json
